# 01 · Tablas Delta — fundamentos

**Objetivo:** entender Delta Lake creando tu primera tabla a partir de datos reales de obras de
construcción de CFE (318 obras, 50 columnas), y ver las capacidades que la diferencian de un
archivo Parquet: transacciones ACID, `MERGE`, *time travel* y metadatos con `DESCRIBE`.

Al terminar sabrás: leer Parquet, escribir una tabla Delta gestionada por Unity Catalog,
actualizar con `MERGE`, y consultar versiones históricas.

In [0]:
%run ./_setup

## 1. Leer los datos de obras desde Parquet
Los datos son **obras de construcción reales de CFE** — 318 registros con información de
avance, importes de contrato, fechas programadas/reales, situación actual, etc.

In [0]:
df_obras = spark.read.parquet(DELTA_LAB_FILE)
print(f"Obras de construcción: {df_obras.count():,}")
df_obras.printSchema()

In [0]:
display(df_obras.summary())

Observa el esquema con 50 campos: identificadores (`IdObra`, `IdProyecto`), importes
(`ImporteContratoMxn`, `ICM`, `CPTT`), avance (`AvanceActualConstruccion`, `AvanceGeneralAP`),
fechas, descripción, problemática, etc. Veamos algunos campos clave:

In [0]:
display(df_obras
 .select("IdObra", "NombreObra", "ImporteContratoMxn",
         "AvanceActualConstruccion", "SituacionActual", "kV"))

## 2. Crear una tabla Delta gestionada
Escribimos las obras como tabla Delta en **tu esquema de trabajo**. Al ser gestionada
por Unity Catalog, su ubicación física la administra Databricks.

In [0]:
(df_obras.write
   .format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(tbl("delta_obras")))

print(f"Tabla creada: {tbl('delta_obras')}")
spark.sql(f"SELECT COUNT(*) AS obras FROM {tbl('delta_obras')}").show()

## 3. `DESCRIBE DETAIL` — metadatos físicos
Número de archivos, tamaño, formato. Útil para diagnosticar *small files* y compresión.

In [0]:
display(spark.sql(f"DESCRIBE DETAIL {tbl('delta_obras')}"))

## 4. Transacciones: `MERGE` (upsert)
Simulamos una actualización: corregir la `SituacionActual` de una obra específica
(por ejemplo, una obra que pasó a "En proceso de terminación").
`MERGE` aplica cambios de forma atómica (ACID) — algo imposible con Parquet plano.

Primero, veamos una obra con datos:

In [0]:
sample_obra = spark.sql(f"""
  SELECT IdObra, NombreObra, SituacionActual, AvanceActualConstruccion
  FROM {tbl('delta_obras')}
  WHERE SituacionActual IS NOT NULL
  ORDER BY ImporteContratoMxn DESC NULLS LAST
  LIMIT 1
""").collect()[0]
print(f"Obra de ejemplo: {sample_obra['NombreObra'][:60]}...")
print(f"  IdObra: {sample_obra['IdObra']}")
print(f"  Situación actual: {sample_obra['SituacionActual'][:80] if sample_obra['SituacionActual'] else 'NULL'}")

In [0]:
# MERGE: actualizar la situación actual de la obra de ejemplo
new_status = "En proceso de terminación - Actualizado en Workshop"
spark.sql(f"""
  MERGE INTO {tbl('delta_obras')} AS t
  USING (SELECT {sample_obra['IdObra']} AS id_obra) AS u
  ON t.IdObra = u.id_obra
  WHEN MATCHED THEN UPDATE SET t.SituacionActual = '{new_status}'
""")

# Verificar
spark.sql(f"""
  SELECT IdObra, SituacionActual
  FROM {tbl('delta_obras')}
  WHERE IdObra = {sample_obra['IdObra']}
""").show(truncate=False)

## 5. Time Travel — historial de versiones
Cada operación crea una versión. Puedes auditar y consultar el pasado.

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {tbl('delta_obras')}"))

In [0]:
# La versión 0 (antes del MERGE) aún tiene la situación original:
before = spark.sql(f"""
  SELECT IdObra, SituacionActual
  FROM {tbl('delta_obras')} VERSION AS OF 0
  WHERE IdObra = {sample_obra['IdObra']}
""")
print("Situación ANTES del MERGE (versión 0):")
before.show(truncate=False)